# Volaris Bot Red Team Notebook

Este notebook agrega un flujo de pruebas para estresar el enrutamiento del bot.
No modifica codigo funcional del servicio.

In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd()
candidates = [cwd, cwd.parent]
project_root = None
for c in candidates:
    if (c / "volaris_teams_bot" / "foundry_client.py").exists():
        project_root = c
        break

if project_root is None:
    raise RuntimeError("No se encontro la raiz del proyecto Volaris-bot.")

os.chdir(project_root)
pkg_dir = str(project_root / "volaris_teams_bot")
if pkg_dir not in sys.path:
    sys.path.insert(0, pkg_dir)

print(f"Project root: {project_root}")
print(f"Python: {sys.executable}")

In [ ]:
import subprocess

cmd = [sys.executable, "volaris_teams_bot/flow_breaker_tests.py"]
res = subprocess.run(cmd, capture_output=True, text=True)

print(res.stdout)
if res.stderr:
    print("[stderr]")
    print(res.stderr)
print(f"Exit code: {res.returncode}")
if res.returncode != 0:
    raise RuntimeError("flow_breaker_tests.py reporto fallas.")

In [ ]:
checklist_path = project_root / "volaris_teams_bot" / "REDTEAM_CHECKLIST.md"
print(checklist_path.read_text(encoding="utf-8"))

In [ ]:
from foundry_client import _infer_single_route, _should_release_pending_route

attack_cases = [
    {
        "name": "Policy question should win",
        "text": "Cuales son los topes para hotel?",
        "last_route": "gastos",
        "pending": "gastos",
    },
    {
        "name": "Pending release by explicit topic switch",
        "text": "Cambiando de tema, cual es la politica de anticipos?",
        "last_route": "gastos",
        "pending": "gastos",
    },
    {
        "name": "Operational expense action",
        "text": "Registra un gasto de taxi por 230 en Orion",
        "last_route": None,
        "pending": None,
    },
    {
        "name": "Anticipo should route to policies",
        "text": "No me gaste todo el anticipo, que hago?",
        "last_route": "gastos",
        "pending": "gastos",
    },
]

for case in attack_cases:
    inferred = _infer_single_route(case["text"], case.get("last_route"))
    release_pending = _should_release_pending_route(case["text"], case.get("pending"))
    print("-" * 70)
    print(f"Case: {case['name']}")
    print(f"Text: {case['text']}")
    print(f"Inferred route: {inferred}")
    print(f"Release pending: {release_pending}")

## Manual stress test (live agents)

Para hacer red-team contra agentes reales desde terminal, ejecuta:

```powershell
python volaris_teams_bot/local_chat.py
```

Luego usa los escenarios de `REDTEAM_CHECKLIST.md`.